# E1.6 · Operating vs outcome guardrails

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.5 · Evaluation output as audit evidence](https://spbreed.github.io/cyber-commons/lessons/E1.5.html)**.

| | |
|---|---|
| Tools used | NeMo Guardrails, LLM Guard, Llama Guard 4, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Classify your own guardrails into the two buckets.

**Why a security engineer needs it.** Frameworks specify how the system works; regulators care what it produced. The control it builds is: constrain both, and know which evidence answers which question.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Two different kinds of control get confused constantly. One bounds how the system runs — budgets, scopes, approvals. The other bounds what it produces. They are tested differently and they fail differently.

> **At CyberTravels.** Two different controls get confused: what bounds how CyberTravels runs (budgets, scopes, approvals) and what bounds what it produces (the hotel recommendation). They fail differently and are tested differently.

## 2 · The framework

```
   operating guardrails            outcome guardrails
   +----------------------+        +-----------------------+
   | HOW it runs          |        | WHAT it produces      |
   | budgets, scopes,     |        | content, decisions,   |
   | approvals, sandbox   |        | actions taken         |
   +----------------------+        +-----------------------+
   tested by attempting    tested by sampling outputs
   the forbidden action    against a rubric

   different tests, different failure modes, constantly confused
```

Guardrails come in two kinds, and confusing them is how a programme passes audit
while missing harm.

**Operating guardrails** constrain *how the system runs*: all egress through the
gateway, privileged tools gated below L3, every action logged. They are testable
today, cheap to verify, and produce clean evidence.

**Outcome guardrails** constrain *what results are acceptable*: no unrecoverable
customer data loss, no increase in customer-facing incidents, no disparate
outcomes across segments. They matter more and most need a measurement you do
not yet have.

The failure is not choosing one. It is shipping only the first column, reporting
it as coverage, and never labelling the second column as unmeasured.

## 3 · The procedure, as a skill

Four operating guardrails are enforceable today; three outcome guardrails are enforceable only where a measurement exists. The skill classifies each rule, specifies the missing measurements, and counts coverage twice — against what shipped and against what was agreed.

In [ ]:
# skills/grc/guardrail-specification/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: guardrail-specification
description: >-
  Separate operating guardrails, which are enforceable today, from outcome
  guardrails, which need a measurement before they mean anything — and specify
  that measurement. Use when a policy contains a rule nobody can enforce.
allowed-tools: Read, Grep, Glob
---

# An outcome guardrail with no measurement is a wish

Operating guardrails constrain what the system may do: which tools, which data,
which actions need approval. They are enforceable today. Outcome guardrails
constrain what the system may cause — no discriminatory decisions, no
misleading advice — and they are enforceable only where somebody has specified
the measurement.

## When to use this

Writing an AI policy, reviewing one, or explaining why a coverage figure of 100%
is counting only the rules that shipped.

## Procedure

**1 — Classify every rule.** Does it constrain the system's behaviour, or the
outcome of that behaviour? The test is whether it can be checked at the moment
of action.

**2 — For each operating guardrail, name the enforcement point.** The gateway,
the tool policy, the approval gate. If there is not one, it is aspirational and
should be reported that way.

**3 — For each outcome guardrail, specify the measurement.** The metric, the
population, the threshold, the cadence, and who reviews it. Four of those five
being present is still not a guardrail.

**4 — Count coverage both ways.** Against the rules that shipped, and against
all the rules that were agreed. The first is usually 100% and the second is
usually about half, and the gap is the honest programme statement.

**5 — Report the unmeasurable ones as open commitments** with an owner and a
date. Leaving them in the policy unmarked is how a policy stops being read.

## Output contract

```json
{
  "rules": [{"text": "str", "kind": "operating|outcome",
             "enforcement_point": "str|null",
             "measurement": {"metric": "str", "population": "str", "threshold": 0.0,
                             "cadence": "str", "reviewer": "str"}}],
  "coverage": {"of_shipped": 1.0, "of_agreed": 0.0},
  "open_commitments": [{"rule": "str", "owner": "str", "due": "str"}]
}
```

## Failure modes

- **Counting only what shipped.** The number is 100% by construction.
- **An outcome guardrail with a metric and no threshold.** Nothing fails.
- **Leaving unmeasurable rules unmarked.** The policy loses credibility as a
  whole.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/grc/guardrail-specification/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/grc/guardrail-specification/scripts/guardrail_specification.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Separate operating guardrails from outcome guardrails and specify the measurement each outcome one needs before it can be enforced.

This is the executable half of the `guardrail-specification` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

def classify(rule, constrains_outcome, measurement_exists):
    kind = "outcome" if constrains_outcome else "operating"
    if kind == "operating":
        return {"rule": rule, "kind": kind, "enforceable_today": True,
                "risk": "may satisfy audit while missing real harm"}
    return {"rule": rule, "kind": kind, "enforceable_today": measurement_exists,
            "risk": ("enforceable" if measurement_exists
                     else "needs an agreed measurement before it can be enforced")}

RULES = [
 ("all agent egress goes through the gateway", False, True),
 ("privileged tools require approval below L3", False, True),
 ("every action is logged with the acting identity", False, True),
 ("agent identities are separately revocable", False, True),
 ("no agent action causes unrecoverable customer data loss", True, False),
 ("automated remediation does not increase customer-facing incidents", True, True),
 ("model outputs do not produce disparate outcomes across segments", True, False),
]
print(f"{'rule':60s}{'kind':11s}{'enforceable':>12}")
print("-" * 86)
for rule, outcome, measurable in RULES:
    c = classify(rule, outcome, measurable)
    print(f"{c['rule']:60s}{c['kind']:11s}{str(c['enforceable_today']):>12}")

operating = [r for r in RULES if not r[1]]
outcome   = [r for r in RULES if r[1]]
enforceable_outcome = [r for r in outcome if r[2]]

print(f"operating guardrails : {len(operating)}  all enforceable today")
print(f"outcome guardrails   : {len(outcome)}  of which enforceable: "
      f"{len(enforceable_outcome)}")

naive = len(operating) / len(RULES)
honest = (len(operating) + len(enforceable_outcome)) / len(RULES)
print(f"\n'guardrail coverage' if you count only what you shipped: "
      f"{len(operating)}/{len(operating)} = 100%")
print(f"coverage across ALL agreed guardrails: "
      f"{len(operating)+len(enforceable_outcome)}/{len(RULES)} = {honest:.0%}")
print("\nThe first number is what usually reaches a steering committee.")

def specify_outcome_guardrail(rule, metric, threshold, source, cadence):
    complete = all([metric, threshold is not None, source, cadence])
    return {"rule": rule, "metric": metric, "threshold": threshold,
            "source": source, "cadence": cadence,
            "status": "enforceable" if complete else "ASPIRATION — label it as such"}

SPECS = [
 specify_outcome_guardrail(
   "automated remediation does not increase customer-facing incidents",
   metric="customer-facing SEV1+SEV2 per 1000 remediations",
   threshold=1.2, source="incident management system", cadence="monthly"),
 specify_outcome_guardrail(
   "no agent action causes unrecoverable customer data loss",
   metric="", threshold=None, source="", cadence=""),
]
for s in SPECS:
    print(f"{s['rule']}")
    print(f"   metric   {s['metric'] or '—'}")
    print(f"   threshold {s['threshold'] if s['threshold'] is not None else '—'}")
    print(f"   source   {s['source'] or '—'}")
    print(f"   status   {s['status']}\n")

def programme_statement(rules, specs):
    enforceable = len([r for r in rules if not r[1]]) + \
                  len([s for s in specs if s["status"] == "enforceable"])
    aspirations = [s["rule"] for s in specs if s["status"] != "enforceable"]
    return (f"{enforceable}/{len(rules)} guardrails are enforceable today.\n"
            f"The following are agreed but UNMEASURED, and are not counted as "
            f"coverage:\n" + "\n".join(f"   - {a}" for a in aspirations))
print(programme_statement(RULES, SPECS))
assert any(s["status"] != "enforceable" for s in SPECS)

## What you just proved

Four operating guardrails are all enforceable today; three outcome guardrails are enforceable only where a measurement exists. Counting only what shipped gives 100% coverage; counting all agreed guardrails gives 71%. One outcome guardrail is fully specified and enforceable; the other is labelled an aspiration and excluded from coverage.

## Your turn

Pick one outcome guardrail your programme has agreed and specify its metric, threshold, source and cadence precisely enough that someone could dispute the result. If you cannot, say so in the coverage report rather than counting it.

---

**Next → [E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*